# Notebook 02: Settlement Detection Model — Fine-Tuning and Inference

**Inputs:** 8-band GeoTIFF composites from Notebook 01, citizen science annotations (Kenya/Uganda), Open Buildings pseudo-labels (Tanzania/Rwanda/Burundi)  
**Outputs:** 3-class settlement probability maps at 10m for all five countries

---

In [ ]:
import json, os
import numpy as np
import torch
import torch.nn as nn
import rasterio
from torch.utils.data import Dataset, DataLoader

with open('../data/config.json') as f:
    CONFIG = json.load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2.1 Install and Load Prithvi Foundation Model


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers', 'huggingface_hub', '-q'])

from huggingface_hub import login

HF_TOKEN = 'huggingface-token-here'

login(token=HF_TOKEN)
print('HuggingFace authenticated.')

## 2.2 Model Architecture: Prithvi Encoder + U-Net Decoder

In [ ]:
from transformers import AutoModel

class SettlementDetector(nn.Module):
    """
    Three-class settlement detector.
    Encoder: Prithvi-100M pretrained transformer (frozen or lightly fine-tuned)
    Decoder: U-Net style upsampling head
    Output classes: 0=background, 1=isolated dwelling, 2=clustered settlement
    """

    def __init__(self, n_classes=3, freeze_encoder=True):
        super().__init__()

        # Load Prithvi encoder
        # the input projection layer is adapted below for 8-band input
        self.encoder = AutoModel.from_pretrained(
            'ibm-nasa-geospatial/Prithvi-100M',
            trust_remote_code=True
        )

        # Adapt input projection from 6 to 8 channels
        # Copy pretrained weights for first 6 channels, initialise remaining 2 from scratch
        old_proj = self.encoder.patch_embed.proj
        new_proj = nn.Conv2d(
            in_channels=8,
            out_channels=old_proj.out_channels,
            kernel_size=old_proj.kernel_size,
            stride=old_proj.stride,
            padding=old_proj.padding,
            bias=(old_proj.bias is not None)
        )
        with torch.no_grad():
            new_proj.weight[:, :6, :, :] = old_proj.weight  # copy pretrained weights
            nn.init.kaiming_normal_(new_proj.weight[:, 6:, :, :])  # init new channels
        self.encoder.patch_embed.proj = new_proj

        # Freeze encoder if specified
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
            # Unfreeze the adapted input projection
            for param in self.encoder.patch_embed.proj.parameters():
                param.requires_grad = True

        # U-Net decoder head
        encoder_dim = 768  # Prithvi-100M hidden dim
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(encoder_dim, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, n_classes, kernel_size=1)
        )

    def forward(self, x):
        # x: (batch, 8, H, W)
        features = self.encoder(x).last_hidden_state  # (batch, seq_len, hidden_dim)
        # Reshape from sequence to spatial
        B, L, C = features.shape
        H = W = int(L ** 0.5)
        features = features.permute(0, 2, 1).reshape(B, C, H, W)
        logits = self.decoder(features)  # (batch, n_classes, H, W)
        return logits


model = SettlementDetector(n_classes=3, freeze_encoder=True).to(DEVICE)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 2.3 Dataset: Class Label Assignment

Assigns one of three classes to each labeled building based on the 50m isolation threshold.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

def assign_settlement_class(gdf, isolation_threshold_m=50):
    """
    Assigns class labels to building footprint GeoDataFrame:
      1 = isolated rural dwelling (no neighbour within isolation_threshold_m)
      2 = clustered settlement (at least one neighbour within isolation_threshold_m)

    Args:
        gdf: GeoDataFrame of building footprint polygons (must be in a metric CRS)
        isolation_threshold_m: distance threshold in metres

    Returns:
        GeoDataFrame with 'settlement_class' column added
    """
    # Project to metric CRS if needed
    if gdf.crs.is_geographic:
        gdf = gdf.to_crs('EPSG:32737')  # UTM zone 37S covers most of study area

    # Compute centroid for each building
    gdf = gdf.copy()
    gdf['centroid'] = gdf.geometry.centroid

    # For each building, check if any other building centroid is within threshold
    centroids = gdf['centroid'].values
    labels = []
    for i, pt in enumerate(centroids):
        others = [c for j, c in enumerate(centroids) if j != i]
        distances = [pt.distance(o) for o in others]
        min_dist = min(distances) if distances else float('inf')
        labels.append(2 if min_dist <= isolation_threshold_m else 1)

    gdf['settlement_class'] = labels
    n_isolated  = labels.count(1)
    n_clustered = labels.count(2)
    print(f'Class assignment complete:')
    print(f'  Isolated dwellings (class 1): {n_isolated:,} ({100*n_isolated/len(labels):.1f}%)')
    print(f'  Clustered settlement (class 2): {n_clustered:,} ({100*n_clustered/len(labels):.1f}%)')
    return gdf


# Load citizen science annotations
annotation_paths = {
    'Kenya':  '../data/annotations/kenya_citizen_science.geojson',
    'Uganda': '../data/annotations/uganda_citizen_science.geojson',
}

annotations = {}
for country, path in annotation_paths.items():
    if os.path.exists(path):
        gdf = gpd.read_file(path)
        gdf = assign_settlement_class(gdf, CONFIG['isolation_threshold_m'])
        annotations[country] = gdf
        print(f'{country}: {len(gdf):,} labeled buildings loaded')
    else:
        print(f'WARNING: {path} not found — download annotations before proceeding')

## 2.4 Pseudo-Label Sampling (Tanzania, Rwanda, Burundi)

Samples Open Buildings detections from the rural stratum only (SMOD class 1),  
filtered to confidence >= 0.75, as pseudo-labels for countries without citizen science data.

In [ ]:
def sample_pseudo_labels(ob_path, stratum_raster_path, country,
                          ob_conf_threshold=0.75, rural_stratum_value=1):
    """
    Loads Open Buildings detections for a country, filters to:
      - rural stratum (SMOD value == 1)
      - confidence >= ob_conf_threshold
    Assigns settlement class using the same 50m isolation threshold.
    Returns GeoDataFrame with 'settlement_class' and 'ob_confidence' columns.

    Args:
        ob_path: path to Open Buildings GeoJSON/Parquet for the country
        stratum_raster_path: path to exported SMOD stratum GeoTIFF
        country: country name string
        ob_conf_threshold: minimum Open Buildings confidence score
        rural_stratum_value: SMOD encoded value for rural (1 in our encoding)
    """
    import rasterio
    from rasterio.sample import sample_gen

    # Load Open Buildings
    ob = gpd.read_file(ob_path)
    print(f'{country}: {len(ob):,} Open Buildings detections loaded')

    # Filter by confidence
    ob = ob[ob['confidence'] >= ob_conf_threshold].copy()
    print(f'{country}: {len(ob):,} remaining after confidence filter >= {ob_conf_threshold}')

    # Sample SMOD stratum at building centroid locations
    ob_proj = ob.to_crs('EPSG:4326')
    coords = [(geom.centroid.x, geom.centroid.y) for geom in ob_proj.geometry]

    with rasterio.open(stratum_raster_path) as src:
        stratum_values = [val[0] for val in src.sample(coords)]
    ob['stratum'] = stratum_values

    # Keep only rural stratum
    ob = ob[ob['stratum'] == rural_stratum_value].copy()
    print(f'{country}: {len(ob):,} remaining after rural stratum filter')

    # Assign settlement class
    ob = assign_settlement_class(ob, CONFIG['isolation_threshold_m'])

    # Store confidence for loss weighting
    ob['sample_weight'] = ob['confidence'] * CONFIG['pseudo_label_discount']

    return ob

pseudo_label_paths = {
    'Tanzania': '../data/open_buildings/tanzania_buildings.geojson',
    'Rwanda':   '../data/open_buildings/rwanda_buildings.geojson',
    'Burundi':  '../data/open_buildings/burundi_buildings.geojson',
}
stratum_raster = '../data/smod_stratum_5country.tif'

pseudo_labels = {}
for country, ob_path in pseudo_label_paths.items():
    if os.path.exists(ob_path) and os.path.exists(stratum_raster):
        pseudo_labels[country] = sample_pseudo_labels(
            ob_path, stratum_raster, country,
            CONFIG['ob_confidence_threshold']
        )
    else:
        print(f'WARNING: data files for {country} not found — download before proceeding')

## 2.5 Loss Function: Focal Loss with Class and Sample Weights

In [ ]:
class WeightedFocalLoss(nn.Module):
    """
    Focal Loss with per-class weights and per-sample weights.

    Class weights (approximate, calibrated during training):
      0 (background):           ~0.05
      1 (isolated dwelling):    ~0.70
      2 (clustered settlement): ~0.55

    Sample weights:
      Citizen science patches:  1.0
      Pseudo-label patches:     ob_confidence * pseudo_label_discount
    """

    def __init__(self, class_weights, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.class_weights = torch.tensor(class_weights, dtype=torch.float)

    def forward(self, logits, targets, sample_weights=None):
        """
        Args:
            logits: (B, C, H, W) raw model output
            targets: (B, H, W) integer class labels
            sample_weights: (B,) per-sample scalar weights, default ones
        """
        B, C, H, W = logits.shape
        class_weights = self.class_weights.to(logits.device)

        # Cross-entropy with class weights
        ce_loss = nn.functional.cross_entropy(
            logits, targets,
            weight=class_weights,
            reduction='none'
        )  # (B, H, W)

        # Focal modulation
        probs = torch.softmax(logits, dim=1)  # (B, C, H, W)
        target_probs = probs.gather(1, targets.unsqueeze(1)).squeeze(1)  # (B, H, W)
        focal_weight = (1 - target_probs) ** self.gamma
        focal_loss = focal_weight * ce_loss  # (B, H, W)

        # Average spatially
        focal_loss = focal_loss.mean(dim=[1, 2])  # (B,)

        # Apply per-sample weights
        if sample_weights is not None:
            sample_weights = sample_weights.to(logits.device)
            focal_loss = focal_loss * sample_weights

        return focal_loss.mean()


# Initialize with approximate class weights (refine after computing class frequencies)
CLASS_WEIGHTS = [0.05, 0.70, 0.55]
criterion = WeightedFocalLoss(class_weights=CLASS_WEIGHTS, gamma=2.0)
print('Loss function initialised.')
print(f'  Class weights: background={CLASS_WEIGHTS[0]}, isolated={CLASS_WEIGHTS[1]}, clustered={CLASS_WEIGHTS[2]}')
print(f'  Focal gamma: 2.0')

## 2.6 Training Loop Scaffold

In [ ]:
# -------------------------------------------------------
# TRAINING HYPERPARAMETERS
# -------------------------------------------------------
LR          = 1e-4
EPOCHS      = 30
BATCH_SIZE  = 8
PATCH_SIZE  = 256  # pixels
# -------------------------------------------------------

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for batch in dataloader:
        images        = batch['image'].to(device)         # (B, 8, H, W)
        labels        = batch['label'].to(device)         # (B, H, W)
        sample_weights = batch['sample_weight'].to(device) # (B,)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels, sample_weights)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)


def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
    return total_loss / len(dataloader), torch.cat(all_preds), torch.cat(all_labels)


print('Training loop scaffold ready.')
print('Build train_loader and val_loader from your Dataset class, then run:')
print('''
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, preds, labels = validate_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")
''')

## 2.7 Discount Factor Ablation

Trains three model variants at d = 0.3, 0.5, 0.7 and selects the best based on Kenya/Uganda held-out validation F1.

In [ ]:
DISCOUNT_FACTORS = [0.3, 0.5, 0.7]
ablation_results = {}

for d in DISCOUNT_FACTORS:
    print(f'\n--- Ablation: discount factor d={d} ---')
    # Update pseudo-label sample weights in dataset
    # (re-instantiate DataLoader with updated weights)
    # train_loader = ... (rebuild with d)
    # model_d = SettlementDetector(...).to(DEVICE)
    # train model_d for EPOCHS ...
    # val_f1 = compute_f1(model_d, val_loader)
    # ablation_results[d] = val_f1
    print(f'  [SCAFFOLD] Instantiate DataLoader with discount_factor={d}, train, evaluate F1')

# best_d = max(ablation_results, key=ablation_results.get)
# print(f'\nBest discount factor: d={best_d} (val F1={ablation_results[best_d]:.4f})')

## 2.8 Save Best Model Checkpoint

In [ ]:
os.makedirs('../models', exist_ok=True)
checkpoint_path = '../models/settlement_detector_best.pt'

# torch.save({
#     'epoch': best_epoch,
#     'model_state_dict': model.state_dict(),
#     'optimizer_state_dict': optimizer.state_dict(),
#     'val_loss': best_val_loss,
#     'config': CONFIG,
# }, checkpoint_path)

print(f'Checkpoint will be saved to: {checkpoint_path}')
print('Uncomment save block above once training is complete.')

---
## Stage 2 Complete

**Files produced:**
- `../models/settlement_detector_best.pt` — trained model checkpoint
- `../outputs/predictions_{country}.tif` — 3-class prediction maps (10m)